# OULAD Graph Pipeline — Canonical Analysis Notebook

**Purpose**: Reproducible record of the Week 8 leakage-safe enrollment-centric graph.

This notebook:
1. Runs the full graph pipeline via `src/graph_pipeline.py`
2. Displays the integrity validation summary
3. Demonstrates `random_student_split` and `lcpo_split` on the enrollment supervision table

> **Note**: GNN training (GraphSAGE) and comparative evaluation are left for the following week.

**Target definition** (fixed throughout this project):
- `1 = at-risk` → Fail or Withdrawn (positive class)
- `0 = success` → Pass or Distinction
- All Precision, Recall, F1, AUPRC values refer to the at-risk class.


## 0. Setup


In [1]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from graph_pipeline import run_pipeline
from oulad_data import random_student_split, lcpo_split
from config import GRAPH_ARTIFACTS_DIR, GRAPH_VALIDATION_DIR

print('Project root:', PROJECT_ROOT)
print('Artifacts dir:', GRAPH_ARTIFACTS_DIR)


Project root: /Users/olivialoza/Documents/Development/OULAD
Artifacts dir: /Users/olivialoza/Documents/Development/OULAD/results/graph/artifacts


## 1. Run the Week 8 Graph Pipeline

Runs all 7 stages: load → filter (cutoff 56 days) → nodes → edges → enrollments → validate → persist.
Assessment filtering uses **due date ≤ 56**, not submission date.


In [2]:
result = run_pipeline(
    week=8,
    data_dir=str(PROJECT_ROOT / 'data' / 'raw'),
    save_dir=str(GRAPH_ARTIFACTS_DIR),
)


  [impute] student: 971 null(s) before imputation — imd_band=971
  [impute] vle_resource: 10486 null(s) before imputation — week_from=5243, week_to=5243


Graph integrity check results:
  dup_nodes_student: 0
  dup_nodes_course_presentation: 0
  dup_nodes_assessment: 0
  dup_nodes_vle_resource: 0
  dup_edges_enrolled_in: 0
  dup_edges_contains_assess: 0
  dup_edges_has_resource: 0
  dup_edges_submitted: 0
  dup_edges_interacted_with: 0
  dangling_enrolled_in_src: 0
  dangling_enrolled_in_dst: 0
  dup_enrollments: 0
  null_values_student: 0
  null_values_course_presentation: 0
  null_values_assessment: 0
  null_values_vle_resource: 0
  label_at_risk: 17208
  label_success: 15385
  label_at_risk_rate: 0.528
Saved 10 artifacts to /Users/olivialoza/Documents/Development/OULAD/results/graph/artifacts/
  nodes_student: week08_nodes_student.parquet
  nodes_course_presentation: week08_nodes_course_presentation.parquet
  nodes_assessment: week08_nodes_assessment.parquet
  nodes_vle_resource: week08_nodes_vle_resource.parquet
  edges_enrolled_in: week08_edges_enrolled_in.parquet
  edges_contains_assess: week08_edges_contains_assess.parquet
  edges

## 2. Graph Statistics


In [3]:
nodes = result['nodes']
edges = result['edges']
enrollments = result['enrollments']

print('=== NODE COUNTS ===')
for ntype, ndf in nodes.items():
    print(f'  {ntype}: {len(ndf):,}')

print('\n=== EDGE COUNTS ===')
for etype, edf in edges.items():
    print(f'  {etype}: {len(edf):,}')

print(f'\n=== ENROLLMENTS ===')
print(f'  Total:   {len(enrollments):,}')
at_risk = int((enrollments["target"] == 1).sum())
print(f'  At-risk: {at_risk:,} ({at_risk/len(enrollments)*100:.1f}%)')
print(f'  Success: {len(enrollments)-at_risk:,} ({(len(enrollments)-at_risk)/len(enrollments)*100:.1f}%)')

print(f'\n=== PERFORMANCE ===')
print(f'  Elapsed:     {result["elapsed_seconds"]:.1f}s')
print(f'  Peak memory: {result["peak_memory_mb"]} MB')


=== NODE COUNTS ===
  student: 28,785
  course_presentation: 22
  assessment: 40
  vle_resource: 6,364

=== EDGE COUNTS ===
  enrolled_in: 32,593
  contains_assess: 40
  has_resource: 6,364
  submitted: 47,259
  interacted_with: 1,056,217

=== ENROLLMENTS ===
  Total:   32,593
  At-risk: 17,208 (52.8%)
  Success: 15,385 (47.2%)

=== PERFORMANCE ===
  Elapsed:     5.1s
  Peak memory: 1048.7 MB


## 3. Validation Summary

Read and display the integrity validation report generated by the pipeline.


In [4]:
validation_summary = GRAPH_VALIDATION_DIR / 'week08_validation_summary.txt'
if validation_summary.exists():
    print(validation_summary.read_text())
else:
    print('Integrity checks from pipeline run:')
    for k, v in result['integrity'].items():
        flag = ' \u26a0' if isinstance(v, int) and k.startswith(('dup_', 'dangling_', 'null_')) and v > 0 else ''
        print(f'  {k}: {v}{flag}')


  OULAD Graph Validation Summary — Week 8

Construction runtime : 6.56 s  |  Peak memory : 1048.7 MB

── Node counts ──────────────────────────────────────────────────
  student                       28,785
  course_presentation               22
  assessment                        40
  vle_resource                   6,364

── Edge counts ──────────────────────────────────────────────────
  enrolled_in                   32,593
  contains_assess                   40
  has_resource                   6,364
  submitted                     47,259
  interacted_with            1,056,217
  enrollments                   32,593

── Duplicates ───────────────────────────────────────────────────
  Duplicate nodes (all types) : 0
  Duplicate edges (all types) : 0
  Duplicate enrollments       : 0

── Dangling edges ───────────────────────────────────────────────
  enrolled_in               src=0  dst=0
  contains_assess           src=0  dst=0
  has_resource              src=0  dst=0
  submitted     

## 4. Label Distribution by Course-Presentation


In [5]:
label_dist = (
    enrollments.groupby(['code_module', 'code_presentation'])['target']
    .agg(total='count', at_risk='sum')
    .assign(at_risk_rate=lambda d: (d['at_risk'] / d['total']).round(3))
    .sort_values('at_risk_rate', ascending=False)
    .reset_index()
)
print(label_dist.to_string(index=False))


code_module code_presentation  total  at_risk  at_risk_rate
        CCC             2014B   1936     1273         0.658
        DDD             2014B   1228      749         0.610
        DDD             2013B   1303      793         0.609
        CCC             2014J   2498     1483         0.594
        DDD             2013J   1938     1109         0.572
        FFF             2014B   1500      846         0.564
        DDD             2014J   1803     1011         0.561
        BBB             2014B   1613      886         0.549
        BBB             2013B   1767      964         0.546
        FFF             2014J   2365     1248         0.528
        BBB             2013J   2237     1165         0.521
        FFF             2013J   2283     1188         0.520
        FFF             2013B   1614      832         0.515
        BBB             2014J   2292     1140         0.497
        EEE             2014B    694      337         0.486
        GGG             2014B    833    

## 5. random_student_split Demo

Demonstrates `random_student_split` from `src/oulad_data.py` on the Week 8 enrollment supervision table.

- Splits are on **unique students** — the same student cannot appear in more than one partition.
- Default fractions: val=10%, test=20%, train=70% of unique students.


In [6]:
train_mask, val_mask, test_mask = random_student_split(
    enrollments, val_frac=0.1, test_frac=0.2, seed=42
)

train_enroll = enrollments[train_mask]
val_enroll   = enrollments[val_mask]
test_enroll  = enrollments[test_mask]

print('=== RANDOM STUDENT SPLIT (enrollment rows) ===')
print(f'  Train: {len(train_enroll):,} rows | {train_enroll["id_student"].nunique():,} unique students')
print(f'  Val:   {len(val_enroll):,} rows | {val_enroll["id_student"].nunique():,} unique students')
print(f'  Test:  {len(test_enroll):,} rows | {test_enroll["id_student"].nunique():,} unique students')

# Verify no student overlap between train and test
train_students = set(train_enroll['id_student'].unique())
test_students  = set(test_enroll['id_student'].unique())
assert train_students.isdisjoint(test_students), 'Student overlap detected!'
print('\n\u2713 No student overlap between train and test.')

# At-risk rates per split
for name, df in [('Train', train_enroll), ('Val', val_enroll), ('Test', test_enroll)]:
    rate = df['target'].mean()
    print(f'  At-risk rate \u2014 {name}: {rate:.3f}')


=== RANDOM STUDENT SPLIT (enrollment rows) ===
  Train: 22,801 rows | 20,150 unique students
  Val:   3,280 rows | 2,878 unique students
  Test:  6,512 rows | 5,757 unique students

✓ No student overlap between train and test.
  At-risk rate — Train: 0.529
  At-risk rate — Val: 0.524
  At-risk rate — Test: 0.526


## 6. LCPO Split Demo

Demonstrates `lcpo_split` from `src/oulad_data.py`.

- For each of the 22 course-presentations, all enrollments in that presentation are held out as test; the rest are train.
- Verifies that all 22 splits yield non-empty train and test sets.


In [7]:
presentations = (
    enrollments[['code_module', 'code_presentation']]
    .drop_duplicates()
    .sort_values(['code_module', 'code_presentation'])
    .reset_index(drop=True)
)

print(f'Course-presentations: {len(presentations)}')
print(f'{"-"*60}')
print(f'{"Module":<12} {"Presentation":<14} {"Train":>8} {"Test":>8} {"Test at-risk%":>14}')
print(f'{"-"*60}')

for _, row in presentations.iterrows():
    train_m, test_m = lcpo_split(enrollments, row.code_module, row.code_presentation)
    test_rate = enrollments.loc[test_m, 'target'].mean()
    print(
        f'{row.code_module:<12} {row.code_presentation:<14}'
        f' {train_m.sum():>8,} {test_m.sum():>8,} {test_rate*100:>13.1f}%'
    )

print(f'\n\u2713 All {len(presentations)} course-presentations yield non-empty train and test splits.')


Course-presentations: 22
------------------------------------------------------------
Module       Presentation      Train     Test  Test at-risk%
------------------------------------------------------------
AAA          2013J            32,210      383          27.4%
AAA          2014J            32,228      365          30.7%
BBB          2013B            30,826    1,767          54.6%
BBB          2013J            30,356    2,237          52.1%
BBB          2014B            30,980    1,613          54.9%
BBB          2014J            30,301    2,292          49.7%
CCC          2014B            30,657    1,936          65.8%
CCC          2014J            30,095    2,498          59.4%
DDD          2013B            31,290    1,303          60.9%
DDD          2013J            30,655    1,938          57.2%
DDD          2014B            31,365    1,228          61.0%
DDD          2014J            30,790    1,803          56.1%
EEE          2013J            31,541    1,052          42.1%

## 7. Node Feature Null Audit

Confirms that all node feature tables have zero nulls after the imputation applied in `build_node_tables()`.


In [8]:
print('Post-imputation null counts per node type:')
for ntype, ndf in nodes.items():
    feat_cols = [c for c in ndf.columns if c != 'node_idx']
    nulls = ndf[feat_cols].isnull().sum()
    total = int(nulls.sum())
    status = '\u2713' if total == 0 else f'\u26a0 {total} nulls'
    print(f'  {ntype}: {status}')
    if total > 0:
        print('   ', nulls[nulls > 0].to_dict())


Post-imputation null counts per node type:
  student: ✓
  course_presentation: ✓
  assessment: ✓
  vle_resource: ✓


---

*Notebook complete. GNN training (GraphSAGE, random-student and LCPO evaluation) follows in the next iteration.*
